# Comparação de métricas: yolov8n.pt vs best.pt

Este notebook compara o modelo base YOLOv8n com o modelo treinado no projeto.


In [2]:
from pathlib import Path
from ultralytics import YOLO
import pandas as pd

base_dir = Path(r'C:\Users\alexa\OneDrive\Área de Trabalho\TCC\Teste Base de Imagens TCC 03')
dataset_yaml = base_dir / 'dataset_yolo' / 'dataset.yaml'
best_model_path = base_dir / 'best.pt'
base_model_path = base_dir / 'yolov8n.pt'

print('dataset_yaml:', dataset_yaml.exists())
print('best.pt:', best_model_path.exists())
print('yolov8n.pt:', base_model_path.exists())


dataset_yaml: True
best.pt: True
yolov8n.pt: True


In [7]:
dataset_yaml.write_text(
    f'path: "{str(base_dir / "dataset_yolo").replace(chr(92), "/")}"\n'
    'train: images/train\n'
    'val: images/val\n'
    'names:\n'
    '  0: "lixo"\n',
    encoding="utf-8"
)

print("YAML corrigido:")
print(dataset_yaml.read_text(encoding="utf-8"))

def get_metrics(model_name: str, model_path: Path):
    if not model_path.exists():
        raise FileNotFoundError(f"Modelo não encontrado: {model_path}")

    model = YOLO(str(model_path))
    metrics = model.val(
        data=str(dataset_yaml),
        imgsz=640,
        batch=8,
        split="val",
        verbose=False,
    )

    precision = float(metrics.box.p)
    recall = float(metrics.box.r)
    map50 = float(metrics.box.map50)
    map50_95 = float(metrics.box.map)

    f1 = 0.0
    if (precision + recall) > 0:
        f1 = float((2 * precision * recall) / (precision + recall))

    results = {
        "modelo": model_name,
        "precision": precision,
        "recall": recall,
        "map50": map50,
        "map50_95": map50_95,
        "f1": f1,
    }

    if hasattr(metrics, "fitness"):
        results["fitness"] = float(metrics.fitness)
    else:
        results["fitness"] = None

    return results

best_metrics = get_metrics("best.pt", best_model_path)
base_metrics = get_metrics("yolov8n.pt", base_model_path)

df = pd.DataFrame([best_metrics, base_metrics])
df = df[["modelo", "precision", "recall", "map50", "map50_95", "f1", "fitness"]]
df = df.sort_values("map50_95", ascending=False)

display(df)

YAML corrigido:
path: "C:/Users/alexa/OneDrive/Área de Trabalho/TCC/Teste Base de Imagens TCC 03/dataset_yolo"
train: images/train
val: images/val
names:
  0: "lixo"

Ultralytics 8.4.182  Python-3.12.2 torch-2.14.0+cpu CPU (AMD Ryzen 7 5700U with Radeon Graphics)
Model summary (fused): 73 layers, 3,132,707 parameters, 0 gradients, 8.6 GFLOPs
val: Fast image access  (ping: 0.2±0.2 ms, read: 63.7±12.5 MB/s, size: 128.4 KB)
val: Scanning C:\Users\alexa\OneDrive\Área de Trabalho\TCC\Teste Base de Imagens TCC 03\dataset_yolo\labels\val... 100 images, 1 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 100/100 361.6it/s 0.3s2s
val: New cache created: C:\Users\alexa\OneDrive\Área de Trabalho\TCC\Teste Base de Imagens TCC 03\dataset_yolo\labels\val.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 13/13 2.1it/s 6.1s0.5s
                   all        100       1590      0.907      0.827      0.867      0.777
Speed: 1.8ms preprocess, 46.3ms

C:\Users\alexa\AppData\Local\Temp\ipykernel_4360\1707655198.py:26: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  precision = float(metrics.box.p)
C:\Users\alexa\AppData\Local\Temp\ipykernel_4360\1707655198.py:27: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  recall = float(metrics.box.r)


val: Fast image access  (ping: 0.0±0.0 ms, read: 424.1±221.9 MB/s, size: 104.6 KB)
val: Scanning C:\Users\alexa\OneDrive\Área de Trabalho\TCC\Teste Base de Imagens TCC 03\dataset_yolo\labels\val.cache... 100 images, 1 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 100/100  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 13/13 2.2it/s 5.9s0.5s
                   all        100       1590     0.0853     0.0994    0.00833    0.00351
Speed: 2.2ms preprocess, 46.6ms inference, 0.0ms loss, 1.5ms postprocess per image
Results saved to C:\Users\alexa\OneDrive\Área de Trabalho\TCC\Teste Base de Imagens TCC 03\runs\detect\val-6


C:\Users\alexa\AppData\Local\Temp\ipykernel_4360\1707655198.py:26: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  precision = float(metrics.box.p)
C:\Users\alexa\AppData\Local\Temp\ipykernel_4360\1707655198.py:27: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  recall = float(metrics.box.r)


,modelo,precision,recall,map50,map50_95,f1,fitness
0,best.pt,0.907477,0.826599,0.866587,0.777182,0.865152,0.777182
1,yolov8n.pt,0.085313,0.099371,0.008329,0.003506,0.091807,0.003506


In [9]:
for item in [best_metrics, base_metrics]:
    print(item["modelo"])
    print("  precision =", round(item["precision"], 4))
    print("  recall    =", round(item["recall"], 4))
    print("  map50     =", round(item["map50"], 4))
    print("  map50_95  =", round(item["map50_95"], 4))
    print("  f1        =", round(item["f1"], 4))

best.pt
  precision = 0.9075
  recall    = 0.8266
  map50     = 0.8666
  map50_95  = 0.7772
  f1        = 0.8652
yolov8n.pt
  precision = 0.0853
  recall    = 0.0994
  map50     = 0.0083
  map50_95  = 0.0035
  f1        = 0.0918
